# Batch Dataloader for Jane Street Parquet Partitions 8 and 9

This notebook builds a memory-efficient batch loader.

Scope:
- Target: `responder_6`
- Weight: `weight`
- Features: all `feature_` columns
- IDs preserved when present: `date_id`, `time_id`, `symbol_id`
- Train partition: partition 8
- Test partition: partition 9



## 1. Inspect repo and data paths

In [1]:
from pathlib import Path

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_ROOT / "data"

print(f"cwd: {cwd}")
print(f"project root: {PROJECT_ROOT}")
print(f"data dir exists: {DATA_DIR.exists()} -> {DATA_DIR}")

if DATA_DIR.exists():
    print("\nNearby data paths:")
    for path in sorted(DATA_DIR.glob("**/*"))[:100]:
        print(path.relative_to(PROJECT_ROOT))

cwd: /Users/akshay/Desktop/js_kaggle/notebooks
project root: /Users/akshay/Desktop/js_kaggle
data dir exists: True -> /Users/akshay/Desktop/js_kaggle/data

Nearby data paths:
data/part_8.parquet
data/part_9.parquet


In [2]:
def resolve_partition_path(partition_id, data_dir=DATA_DIR):
    """Resolve expected Kaggle partition folders or nearby local parquet files."""
    candidates = [
        data_dir / "train.parquet" / f"partition_id={partition_id}",
        data_dir / "train.parquet" / f"partition_id={partition_id}.parquet",
        data_dir / f"partition_id={partition_id}",
        data_dir / f"partition_id={partition_id}.parquet",
        data_dir / f"part_{partition_id}.parquet",
    ]
    existing = [path for path in candidates if path.exists()]
    if not existing:
        candidate_text = "\n".join(str(path) for path in candidates)
        raise FileNotFoundError(
            f"Could not find partition {partition_id}. Checked:\n{candidate_text}"
        )
    return existing[0]


TRAIN_PATH = resolve_partition_path(8)
TEST_PATH = resolve_partition_path(9)

print(f"train partition path: {TRAIN_PATH}")
print(f"test partition path:  {TEST_PATH}")

train partition path: /Users/akshay/Desktop/js_kaggle/data/part_8.parquet
test partition path:  /Users/akshay/Desktop/js_kaggle/data/part_9.parquet


## 2. Imports

In [3]:
import pyarrow.dataset as ds
import pandas as pd
import numpy as np

try:
    import polars as pl
    POLARS_AVAILABLE = True
except ImportError:
    pl = None
    POLARS_AVAILABLE = False

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"polars available: {POLARS_AVAILABLE}")

pandas: 2.2.2
numpy: 1.26.4
polars available: False


## 3. Inspect parquet schema only

In [4]:
TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
ID_CANDIDATES = ["date_id", "time_id", "symbol_id"]


def parquet_schema(parquet_path):
    dataset = ds.dataset(parquet_path, format="parquet")
    return dataset.schema


train_schema = parquet_schema(TRAIN_PATH)
test_schema = parquet_schema(TEST_PATH)

train_columns = train_schema.names
test_columns = test_schema.names

FEATURE_COLS = [col for col in train_columns if col.startswith("feature_")]
ID_COLS = [col for col in ID_CANDIDATES if col in train_columns]

print(f"train column count: {len(train_columns)}")
print(f"test column count:  {len(test_columns)}")
print("\nTrain columns:")
print(train_columns)

print(f"\nfeature column count: {len(FEATURE_COLS)}")
print(f"first 10 feature columns: {FEATURE_COLS[:10]}")
print(f"{TARGET_COL} exists in train: {TARGET_COL in train_columns}")
print(f"{TARGET_COL} exists in test:  {TARGET_COL in test_columns}")
print(f"{WEIGHT_COL} exists in train: {WEIGHT_COL in train_columns}")
print(f"{WEIGHT_COL} exists in test:  {WEIGHT_COL in test_columns}")
print(f"ID columns present: {ID_COLS}")

missing_required_train = [col for col in [TARGET_COL, WEIGHT_COL] if col not in train_columns]
missing_required_test = [col for col in [TARGET_COL, WEIGHT_COL] if col not in test_columns]
if missing_required_train:
    raise ValueError(f"Missing required train columns: {missing_required_train}")
if missing_required_test:
    raise ValueError(f"Missing required test columns: {missing_required_test}")
if not FEATURE_COLS:
    raise ValueError("No feature_ columns found in train schema")

train column count: 92
test column count:  92

Train columns:
['date_id', 'time_id', 'symbol_id', 'weight', 'feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04', 'feature_05', 'feature_06', 'feature_07', 'feature_08', 'feature_09', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42', 'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47', 'feature_48', 'feature_49', 'feature_50', 'feature_51', 'feature_52', 'feature_53', 'feature_54', 'feature_55', 'feature_56', 'feature_57', 'feature_58', 'feature_59', 'feature_60', 'feature_61', 'feature_62', 'feature_6

## 4. Reusable batch loader

In [5]:
def batch_loader(
    parquet_path,
    feature_cols,
    batch_size,
    target_col=TARGET_COL,
    weight_col=WEIGHT_COL,
    id_cols=ID_CANDIDATES,
):
    """
    Stream selected parquet columns in batches.

    Yields:
        X: np.ndarray, shape (n_rows, n_features), float32
        y: np.ndarray, shape (n_rows,), float32
        weights: np.ndarray, shape (n_rows,), float32
        ids: pd.DataFrame with available ID/time columns
    """
    parquet_path = Path(parquet_path)
    dataset = ds.dataset(parquet_path, format="parquet")
    available_cols = set(dataset.schema.names)

    selected_features = [col for col in feature_cols if col in available_cols]
    selected_ids = [col for col in id_cols if col in available_cols]
    required_cols = [target_col, weight_col]
    missing_required = [col for col in required_cols if col not in available_cols]

    if missing_required:
        raise ValueError(f"Missing required columns in {parquet_path}: {missing_required}")
    if not selected_features:
        raise ValueError(f"No requested feature columns found in {parquet_path}")

    columns_to_read = selected_ids + selected_features + required_cols
    scanner = dataset.scanner(columns=columns_to_read, batch_size=batch_size)

    for record_batch in scanner.to_batches():
        batch_df = record_batch.to_pandas()

        batch_df = batch_df.dropna(subset=[target_col, weight_col])
        if batch_df.empty:
            continue

        ids = batch_df[selected_ids].reset_index(drop=True) if selected_ids else pd.DataFrame(index=range(len(batch_df)))
        features = batch_df[selected_features].fillna(0)

        X = features.to_numpy(dtype=np.float32, copy=False)
        y = batch_df[target_col].to_numpy(dtype=np.float32, copy=False)
        weights = batch_df[weight_col].to_numpy(dtype=np.float32, copy=False)

        yield X, y, weights, ids

## 5. Test one batch from partition 8 and partition 9

In [6]:
BATCH_SIZE = 65_536


def inspect_one_batch(name, parquet_path):
    loader = batch_loader(parquet_path, FEATURE_COLS, batch_size=BATCH_SIZE)
    X, y, weights, ids = next(loader)
    missing_counts = {
        "target_missing_after": int(pd.isna(y).sum()),
        "weight_missing_after": int(pd.isna(weights).sum()),
        "feature_missing_after": int(np.isnan(X).sum()),
    }

    print(f"{name} path: {parquet_path}")
    print(f"X shape:       {X.shape}")
    print(f"y shape:       {y.shape}")
    print(f"weights shape: {weights.shape}")
    print(f"ids shape:     {ids.shape}")
    print("\nFirst few ID rows:")
    display(ids.head())
    print("\nMissing value counts after cleaning:")
    print(missing_counts)
    return X, y, weights, ids


train_batch = inspect_one_batch("partition 8", TRAIN_PATH)

partition 8 path: /Users/akshay/Desktop/js_kaggle/data/part_8.parquet
X shape:       (65536, 79)
y shape:       (65536,)
weights shape: (65536,)
ids shape:     (65536, 3)

First few ID rows:


,date_id,time_id,symbol_id
0,1360,0,0
1,1360,0,1
2,1360,0,2
3,1360,0,3
4,1360,0,4



Missing value counts after cleaning:
{'target_missing_after': 0, 'weight_missing_after': 0, 'feature_missing_after': 0}


In [7]:
test_batch = inspect_one_batch("partition 9", TEST_PATH)

partition 9 path: /Users/akshay/Desktop/js_kaggle/data/part_9.parquet
X shape:       (65536, 79)
y shape:       (65536,)
weights shape: (65536,)
ids shape:     (65536, 3)

First few ID rows:


,date_id,time_id,symbol_id
0,1530,0,0
1,1530,0,1
2,1530,0,2
3,1530,0,3
4,1530,0,4



Missing value counts after cleaning:
{'target_missing_after': 0, 'weight_missing_after': 0, 'feature_missing_after': 0}


## 6. Collect limited batches for small experiments

In [8]:
def collect_batches(loader, max_rows):
    """Collect at most max_rows from a batch loader into memory for small experiments."""
    X_parts = []
    y_parts = []
    weight_parts = []
    rows_collected = 0

    for X, y, weights, ids in loader:
        remaining = max_rows - rows_collected
        if remaining <= 0:
            break

        take = min(len(y), remaining)
        X_parts.append(X[:take])
        y_parts.append(y[:take])
        weight_parts.append(weights[:take])
        rows_collected += take

        if rows_collected >= max_rows:
            break

    if not X_parts:
        return (
            np.empty((0, 0), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
        )

    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    weights = np.concatenate(weight_parts, axis=0)
    return X, y, weights

In [9]:
MAX_ROWS = 100_000

small_train_loader = batch_loader(TRAIN_PATH, FEATURE_COLS, batch_size=BATCH_SIZE)
X_small, y_small, weights_small = collect_batches(small_train_loader, max_rows=MAX_ROWS)

print(f"Collected rows: {len(y_small):,}")
print(f"X_small shape:       {X_small.shape}")
print(f"y_small shape:       {y_small.shape}")
print(f"weights_small shape: {weights_small.shape}")
print(f"X missing values: {int(np.isnan(X_small).sum())}")
print(f"y missing values: {int(np.isnan(y_small).sum())}")
print(f"weight missing values: {int(np.isnan(weights_small).sum())}")

Collected rows: 100,000
X_small shape:       (100000, 79)
y_small shape:       (100000,)
weights_small shape: (100000,)
X missing values: 0
y missing values: 0
weight missing values: 0


In [10]:
small_test_loader = batch_loader(TEST_PATH, FEATURE_COLS, batch_size=BATCH_SIZE)
X_small_test, y_small_test, weights_small_test = collect_batches(small_test_loader, max_rows=MAX_ROWS)

print(f"Collected rows: {len(y_small_test):,}")
print(f"X_small_test shape:       {X_small_test.shape}")
print(f"y_small_test shape:       {y_small_test.shape}")
print(f"weights_small_test shape: {weights_small_test.shape}")
print(f"X missing values: {int(np.isnan(X_small_test).sum())}")
print(f"y missing values: {int(np.isnan(y_small_test).sum())}")
print(f"weight missing values: {int(np.isnan(weights_small_test).sum())}")

Collected rows: 100,000
X_small_test shape:       (100000, 79)
y_small_test shape:       (100000,)
weights_small_test shape: (100000,)
X missing values: 0
y missing values: 0
weight missing values: 0
